In [1]:

%pip install pandas
%pip install ollama



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
#borrowing this code from Leon 

import os
import glob
import json
import gzip
import time 

import pandas as pd
from ollama import chat
from pydantic import BaseModel

In [3]:
## importing the files from the transcripts export folder
from pathlib import Path


def resolve_transcript_folder():
    candidates = [
        Path("transcripts"),
        Path("assignment 1/transcripts"),
        Path("../assignment 1/transcripts"),
    ]
    for folder in candidates:
        if folder.exists():
            return str(folder)
    return str(candidates[0])


ed_transcripts = resolve_transcript_folder()
print(f"Using transcripts folder: {ed_transcripts}")


def load_transcript(path):
    with open(path, encoding="utf-8") as f:
        raw = f.read()

    # header and body are separated by the first blank line
    header, _, body = raw.partition("\n\n")

    meta = {}
    for line in header.splitlines():
        key, _, value = line.partition(": ")
        meta[key.strip().lower()] = value.strip()

    return {
        "title": meta.get("title", ""),
        "date": meta.get("date", ""),
        "subcommittee": meta.get("subcommittee", ""),
        "url": meta.get("url", ""),
        "video_id": os.path.splitext(os.path.basename(path))[0].split("_")[-1],
        "file": path,
        "text": body.strip(),
    }


def load_all(folder=ed_transcripts):
    paths = sorted(glob.glob(os.path.join(folder, "*.txt")))
    if not paths:
        print(f"No transcript .txt files found in: {folder}")
        return pd.DataFrame(columns=[
            "title", "date", "subcommittee", "url", "video_id", "file", "text", "word_count"
        ])

    df = pd.DataFrame([load_transcript(p) for p in paths])
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["word_count"] = df["text"].str.split().str.len()
    return df.sort_values("date").reset_index(drop=True)


df = load_all()
if df.empty:
    print("0 transcripts loaded")
else:
    print(df[["date", "title", "subcommittee", "word_count"]])
    print(f"\n{len(df)} transcripts loaded")

Using transcripts folder: transcripts
        date                                              title  \
0 2026-01-14                     "Building an AI-Ready America"   
1 2026-02-11  "Building an AI-Ready America: Safer Workplace...   
2 2026-02-24  "Building an AI-Ready America: Teaching in the...   
3 2026-03-04  "Building an AI-Ready America: Strengthening E...   
4 2026-04-15  "Building an AI-Ready America: Understanding A...   
5 2026-06-03  "Building an AI-Ready America: Higher Educatio...   

                                        subcommittee  word_count  
0                             Full Committee Hearing       29998  
1              Subcommittee on Workforce Protections       10739  
2  Subcommittee on Early Childhood, Elementary, a...       16903  
3  Subcommittee on Higher Education and Workforce...       16137  
4              Subcommittee on Workforce Protections       10841  
5  Subcommittee on Higher Education and Workforce...       17235  

6 transcripts loaded


In [6]:
##stealing this code from Leon
import re

keywords_ai = [
    "artificial intelligence",
    "AI",
    "AI-assisted",
    "large language model",
    "large language models",
    "GenAI",
    "generative AI",
    "generative artificial intelligence",
    "LLM",
    "LLMs",
]

# escape + word boundaries to avoid weird partial matches
keyword_pattern = re.compile(
    r"\b(?:" + "|".join(re.escape(k) for k in keywords_ai) + r")\b",
    flags=re.IGNORECASE,
)

if "text" not in df.columns:
    raise ValueError("Expected a 'text' column on df. Re-run Cell 3 first.")

df["text"] = df["text"].fillna("")
df["keyword_hits"] = df["text"].apply(lambda t: keyword_pattern.findall(t))
df["keyword_hit_count"] = df["keyword_hits"].str.len()
df["contains_keywords_ai"] = df["keyword_hit_count"] > 0

print(f"Scanned {len(df)} transcripts")
print(f"Total keyword mentions found: {int(df['keyword_hit_count'].sum())}")
print(f"Transcripts with >=1 mention: {int(df['contains_keywords_ai'].sum())}")

# quick QA view: highest-hit transcripts
if len(df) > 0:
    display(
        df[["date", "title", "keyword_hit_count", "contains_keywords_ai"]]
        .sort_values("keyword_hit_count", ascending=False)
        .head(10)
    )

Scanned 6 transcripts
Total keyword mentions found: 1315
Transcripts with >=1 mention: 6


,date,title,keyword_hit_count,contains_keywords_ai
0,2026-01-14,"""Building an AI-Ready America""",436,True
5,2026-06-03,"""Building an AI-Ready America: Higher Educatio...",247,True
2,2026-02-24,"""Building an AI-Ready America: Teaching in the...",233,True
4,2026-04-15,"""Building an AI-Ready America: Understanding A...",181,True
1,2026-02-11,"""Building an AI-Ready America: Safer Workplace...",118,True
3,2026-03-04,"""Building an AI-Ready America: Strengthening E...",100,True


In [10]:
## Run each transcript through Ollama and save one JSON label per transcript
MODEL_NAME = "qwen3.5:4b"  # primary model
FALLBACK_MODEL = "qwen2.5:1.5b"  # optional smaller fallback (pull if needed)
MAX_CHARS = 12000  # let model see more of each transcript
MAX_QUOTES = 5
OVERWRITE_EXISTING = True  # set False to keep existing JSON files

class AILabel(BaseModel):
    """Structured response from the model."""
    contains_ai_llm: bool
    quotes: list[str]


dir_labels = "output/ai_labels"
os.makedirs(dir_labels, exist_ok=True)


def run_labeling(prompt: str):
    """Try GPU first, then CPU mode, then smaller model."""
    attempts = [
        (MODEL_NAME, {"num_ctx": 2048}, "primary-gpu"),
        (MODEL_NAME, {"num_ctx": 1024, "num_gpu": 0}, "primary-cpu"),
        (FALLBACK_MODEL, {"num_ctx": 2048}, "fallback-gpu"),
        (FALLBACK_MODEL, {"num_ctx": 1024, "num_gpu": 0}, "fallback-cpu"),
    ]

    errors = []
    for model_name, options, mode in attempts:
        try:
            response = chat(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                format=AILabel.model_json_schema(),
                options=options,
            )
            return AILabel.model_validate_json(response.message.content), model_name, mode
        except Exception as e:
            errors.append(f"{mode} ({model_name}): {e}")

    raise RuntimeError(" | ".join(errors))


for _, row in df.iterrows():
    fn_out = os.path.join(dir_labels, f"{row['video_id']}.json")

    if os.path.exists(fn_out) and not OVERWRITE_EXISTING:
        continue

    transcript_text = (row.get("text") or "").strip()
    if not transcript_text:
        continue

    transcript_text = transcript_text[:MAX_CHARS]
    prompt = f"""Read the hearing transcript below and decide if it clearly mentions AI or LLMs.
Return valid JSON with exactly these keys:
- contains_ai_llm (boolean)
- quotes (array of up to {MAX_QUOTES} short direct quotes from transcript; [] if no mention)

Rules:
- quotes must be verbatim text snippets from the transcript.
- prefer distinct quotes from different parts of the transcript.
- do not include explanations outside JSON.

Transcript:
{transcript_text}
"""

    try:
        label, used_model, run_mode = run_labeling(prompt)

        clean_quotes = []
        for q in (label.quotes or []):
            q2 = (q or "").strip()
            if q2 and q2 not in clean_quotes:
                clean_quotes.append(q2)
        clean_quotes = clean_quotes[:MAX_QUOTES]

        record = {
            "video_id": row["video_id"],
            "date": str(row["date"].date()) if pd.notna(row["date"]) else None,
            "title": row["title"],
            "model": used_model,
            "run_mode": run_mode,
            "contains_ai_llm": label.contains_ai_llm,
            "quote_count": len(clean_quotes),
            "quotes": clean_quotes,
            "quote": clean_quotes[0] if clean_quotes else "",  # backward compatibility
        }

        with open(fn_out, "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        print(f"Saved {row['video_id']} with {used_model} ({run_mode}) - {len(clean_quotes)} quote(s)")

    except Exception as e:
        print(f"Failed on {row['video_id']}: {e}")

Saved RM0aq5ynUiQ with qwen3.5:4b (primary-gpu) - 5 quote(s)
Saved Wd9cdenGMZI with qwen3.5:4b (primary-gpu) - 5 quote(s)
Saved ZeJBjQY3LdI with qwen3.5:4b (primary-gpu) - 5 quote(s)
Saved 8jANpuMrFG8 with qwen3.5:4b (primary-gpu) - 4 quote(s)
Saved 7QljNpZdu3Q with qwen3.5:4b (primary-gpu) - 5 quote(s)
Saved kfRYHhh8yD4 with qwen3.5:4b (primary-gpu) - 5 quote(s)


In [ ]:
labels = []
for fn in glob.glob(os.path.join(dir_labels, "*.json")):
    with open(fn, encoding="utf-8") as f:
        labels.append(json.load(f))

if not labels:
    print("No label files found yet. Run the Ollama labeling cell first.")
else:
    df_labels = pd.DataFrame(labels)

    # backward compatibility across old/new label formats
    if "quotes" not in df_labels.columns:
        df_labels["quotes"] = [[] for _ in range(len(df_labels))]
    if "quote" not in df_labels.columns:
        df_labels["quote"] = ""

    df_labels["quote_count"] = df_labels["quotes"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

    print(df_labels["contains_ai_llm"].value_counts(dropna=False))
    print("\nQuote count summary:")
    print(df_labels["quote_count"].describe())

    # Optional: join back to transcript metadata
    cols = ["video_id", "contains_ai_llm", "quote", "quotes", "quote_count"]
    if "model" in df_labels.columns:
        cols.append("model")

    df_with_labels = df.merge(
        df_labels[cols],
        on="video_id",
        how="left",
    )
    df_with_labels

contains_ai_llm
True    6
Name: count, dtype: int64


In [9]:
# Compare keyword detection vs Ollama labels at transcript level
if "contains_keywords_ai" not in df.columns:
    if "keywords_ai" not in globals():
        keywords_ai = [
            "artificial intelligence",
            "AI",
            "AI-assisted",
            "large language models",
            "GenAI",
            "generative AI",
            "generative artificial intelligence",
        ]
    df["contains_keywords_ai"] = df["text"].str.contains(
        "|".join(keywords_ai), regex=True, case=False
    )

if "df_labels" not in globals() or df_labels.empty:
    print("Run Cell 6 first to build df_labels.")
else:
    compare_cols = ["video_id", "contains_ai_llm"]
    df_compare = df.merge(
        df_labels[compare_cols].drop_duplicates(subset=["video_id"]),
        on="video_id",
        how="left",
    )

    pd.crosstab(
        df_compare["contains_keywords_ai"],
        df_compare["contains_ai_llm"],
        rownames=["Keyword hit"],
        colnames=["Ollama label"],
        margins=True,
        dropna=False,
    )



Run Cell 6 first to build df_labels.


In [9]:
fn_to_annotate = "output/ai_to_annotate.csv"

if "contains_keywords_ai" not in df.columns:
    if "keywords_ai" not in globals():
        keywords_ai = [
            "artificial intelligence",
            "AI",
            "AI-assisted",
            "large language models",
            "GenAI",
            "generative AI",
            "generative artificial intelligence",
        ]
    df["contains_keywords_ai"] = df["text"].str.contains(
        "|".join(keywords_ai), regex=True, case=False
    )

# KEYWORD-ONLY candidates 
df_to_annotate = df[df["contains_keywords_ai"].fillna(False)].copy()

# keep this file lightweight (no full transcript text column)
cols = [
    "video_id",
    "date",
    "title",
    "contains_keywords_ai",
]
df_to_annotate[cols].to_csv(fn_to_annotate, index=False)
print(f"Saved {len(df_to_annotate)} keyword-hit transcripts to {fn_to_annotate}")

Saved 6 keyword-hit transcripts to output/ai_to_annotate.csv


Ok this is the code Leon made to have his little yes/no pop up window, I haven't been able to run it because my videos have like 100,000 mentions of AI (naturally) so its not really helpful for me at this point. But! you should try it 

In [ ]:
import csv
import os
import random
import sys

fn_items = "output/ai_to_annotate.csv"
fn_labels = "output/ai_human_labels.csv"

SEED = 303
fieldnames = ["video_id", "contains_ai_human", "interesting", "reviewer", "note"]
answers = {"t": "True", "f": "False", "s": ""}

reviewer = input("your name > ").strip()
video_filter = input("video_id to review (blank = all) > ").strip()

# one row per transcript, shuffled reproducibly
with open(fn_items, encoding="utf-8", newline="") as f:
    items = list(csv.DictReader(f))
random.Random(SEED).shuffle(items)

if video_filter:
    items = [item for item in items if item.get("video_id") == video_filter]
    if not items:
        print(f"No rows found for video_id={video_filter}")

# skip transcripts this reviewer already labeled
done = set()
if os.path.exists(fn_labels):
    with open(fn_labels, encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            if row.get("reviewer") == reviewer:
                done.add(row.get("video_id"))

queue = [item for item in items if item.get("video_id") not in done]
print(f"{len(items) - len(queue)} of {len(items)} already labeled by {reviewer}")

os.makedirs("output", exist_ok=True)
is_new = not os.path.exists(fn_labels)
f = open(fn_labels, "a", newline="", encoding="utf-8")
writer = csv.DictWriter(f, fieldnames=fieldnames)
if is_new:
    writer.writeheader()
    f.flush()

# helper: pull transcript text from df when ai_to_annotate.csv is lightweight
def get_text_for_video(video_id: str) -> str:
    if "df" in globals() and "video_id" in df.columns and "text" in df.columns:
        rows = df.loc[df["video_id"] == video_id, "text"]
        if len(rows) > 0:
            return (rows.iloc[0] or "").strip()
    return ""

history = []  # (position in queue, file size before writing) for undo
i = 0
while i < len(queue):
    item = queue[i]
    video_id = item.get("video_id", "")
    url = f"https://www.youtube.com/watch?v={video_id}"

    print("\n" + "-" * 70)
    print(f"{len(items) - len(queue) + i + 1} of {len(items)}  |  {url}")
    print(f"TITLE: {item.get('title', '').strip()}")

    text = (item.get("text") or "").strip()
    if not text:
        text = get_text_for_video(video_id)
    print(text[:1800] + ("\n...[truncated]" if len(text) > 1800 else ""))
    print("-" * 70)

    reply = input("mentions AI/LLMs? [t/f/s, add i = interesting, b = undo, q = quit] > ").strip()
    key, _, note = reply.partition(" ")
    key = key.lower()

    interesting = "i" in key and key != "i"
    key = key.replace("i", "") if interesting else key

    if key == "q":
        break
    if key == "b":
        if history:
            i, size = history.pop()
            f.truncate(size)
            f.seek(0, os.SEEK_END)
        else:
            print("nothing to undo")
        continue
    if key not in answers:
        print("answer t, f, s (add i for interesting), b or q")
        continue

    history.append((i, f.tell()))
    writer.writerow({
        "video_id": video_id,
        "contains_ai_human": answers[key],
        "interesting": "True" if interesting else "",
        "reviewer": reviewer,
        "note": note,
    })
    f.flush()
    i += 1

f.close()
print(f"labels saved to {fn_labels}")

This code is for comparing annotations by different people, guess we need the above code to run first? 

In [ ]:
# Load human AI labels (transcript-level) and compare across reviewers
import csv
import sys

# allow very long text fields in CSV
max_int = sys.maxsize
while True:
    try:
        csv.field_size_limit(max_int)
        break
    except OverflowError:
        max_int = int(max_int / 10)

fn_human_labels = "output/ai_human_labels.csv"

if not os.path.exists(fn_human_labels):
    print(f"No labels file found at {fn_human_labels}. Run Cell 9 first.")
else:
    df_annotations = pd.read_csv(fn_human_labels, dtype=str, keep_default_na=False)

    if df_annotations.empty:
        print("Labels file is empty.")
    else:
        # normalize values
        df_annotations["contains_ai_human"] = df_annotations["contains_ai_human"].replace("", "skip")

        # one row per transcript, one column per reviewer
        df_wide = df_annotations.pivot_table(
            index=["video_id"],
            columns="reviewer",
            values="contains_ai_human",
            aggfunc="first",
            fill_value="skip",
        )

        col_reviewers = df_wide.columns.tolist()
        print(f"Loaded {len(df_annotations)} human labels from {len(col_reviewers)} reviewer(s).")
        df_wide.head()

In [ ]:
has_skip = (df_wide[col_reviewers] == "skip").any(axis=1)
has_same_answer = df_wide[col_reviewers].nunique(axis=1) == 1
df_wide["is_unanimous"] = has_same_answer & ~has_skip

answered = df_wide[col_reviewers].isin(["True", "False"]).sum()
share_true = (df_wide[col_reviewers] == "True").sum() / answered

print(f"Reviewers agreed on {df_wide['is_unanimous'].mean():.0%} of segments")
print(f"Between {share_true.min():.0%} and {share_true.max():.0%} of segments mention voting, depending on the reviewer")

In [ ]:
df_text = pd.read_csv(fn_to_annotate, dtype=str).set_index(["video_id", "id"])
df_review = df_wide.loc[~df_wide["is_unanimous"], col_reviewers].join(df_text["text"])

display_annotator_diffs(df_review.head(3), col_reviewers)